Notebook created by [Peter Belcák](https://disco.ethz.ch/members/pbelcak)

Updated:<br>

For FS24 by [Andreas Plesner](https://disco.ethz.ch/members/aplesner)

For HS24, FS25, HS25, FS26 by [Till Aczel](https://disco.ethz.ch/members/taczel)


In [5]:
# Uncomment for Colab
# import os
# os.makedirs("data_scratch", exist_ok=True)

# Uncomment for Snowflake
# import os
# import getpass
# user = getpass.getuser()
# scratch_dir = os.path.join("/scratch", user)
# os.makedirs(scratch_dir, exist_ok=True)

# Hand-On Deep Learing - Introduction Session

This session will introduce you to the basic concepts of differentiable programming and traning neural networks from scratch. We will be using the popular deep learning framework PyTorch.


We will first talk about the bread-and-butter data type of deep learning - tensors. Once done, we will give a quick introduction to differentiable programs. On a simple example, we will show how to compute gradients in pytorch, and then lead you towards an implementation of a simple gradient descent training loop.

Having got a better feel of how one trains the parameters of a differentiable program, we will introduce neural networks. We will provide you with a full implementation of a training loop, and help you train an approximation of a Boolean function.

We will then steer away from illustrative examples and get started with practical, application-oriented deep learning. We will define and train both shallow and deep neural networks for image classification, tune the parameters of training and the sizes of architectures, and observe how the individual properties of our training setup influence the quality of the learning outcomes.

We will conclude this session with the introduction of convolutional layers. The challenge of the day will be to use convolutional neural networks to correctly classify greyscale images of fashion articles.

## Prelude: Tensors

In [6]:
import torch
import torch.nn as nn

For training of deep models, `torch` uses a special data type: `torch.tensor`. The `torch.tensor` is `torch`'s way to store tensors which can be seen as multidimensional arrays, with the vector and matrix simply being the 1 and 2 dimensional instances. The values in `torch.tensor` can be learned from data.

Before we can move on to do anything more exciting, one has to know a bit about tensors. Anyone familiar with `numpy.ndarray` can skip to the last subsection on trainable tensors.

#### Tensors are an enhanced, uniform variant of multi-dimensional lists that torch operations can eat.

In [7]:
A = [[ 0.0, 1.0], [ 1.0 , 0.0]]

try:
  torch.matmul(A, A) # throws a TypeError
except TypeError as error:
  print(f"An error occured in {error}")

An error occured in matmul(): argument 'input' (position 1) must be Tensor, not list


In [8]:
A_tensor = torch.tensor(A)

torch.matmul(A_tensor, A_tensor)

tensor([[1., 0.],
        [0., 1.]])

Notice that nested lists that are candidates for tensors must be uniform in every dimension

In [9]:
B = [
    [[1, 2, 3], [4, 5, 6]],
    [[0, 0], [1, 1]]
]


try:
  B_tensor = torch.tensor(B) # throws a ValueError
except ValueError as error:
  print(f"Could not form a tensor: {error}")

Could not form a tensor: expected sequence of length 3 at dim 2 (got 2)


**Exercise.** Create a tensor `I_tensor`, which is a 3x3 identity matrix.

In [10]:
# Write your code here

I = [[1,0,0], [0,1,0], [0,0,1]]

I_tensor = torch.tensor(I) 


#### Tensors have a multi-dimensional `size`, also known as `shape`
The shape of a tensor describes the sizes of its individual tensor dimensions (also known as *axes*).

In [11]:
A_tensor.shape

torch.Size([2, 2])

In [12]:
A_tensor.size()

torch.Size([2, 2])

Notice that `A` consists of two lists containing two elements each.
Correspondingly, `A_tensor` has size of `[2, 2]`, meaning that `A_tensor` consists of to sub-tensors, namely `A_tensor[0]` and `A_tensor[1]`, containing two elements each.

In [13]:
A_tensor[0]

tensor([0., 1.])

If we take `B` such that `B` contains two lists of lists, such as in

In [14]:
B = [
    [[1, 2, 3], [4, 5, 6]],
    [[0, 0, 0], [1, 1, 1]]
]

Then we can turn it into a `B_tensor` of size `[2,2,3]`,

In [15]:
B_tensor = torch.tensor(B)
B_tensor.shape

torch.Size([2, 2, 3])

... meaning that `B_tensor` consits of two two-dimensional sub-tensors, `B_tensor[0]`, and `B_tensor[1]`.

**Exercise.** What should be the shape of `I_tensor`?

In [16]:
I_tensor_shape_intended = torch.Size( [3,3  ] ) # modify this line with your guess

**Exercise.** Retrieve the shape of `I_tensor`.

In [17]:
I_tensor_shape = I_tensor.shape # modify this line with the correct code

**Exercise.** Are they the same?

In [18]:
# Run this code block.

if I_tensor_shape_intended == I_tensor_shape:
  print("I_tensor has shape as intended.")
else:
  print("I_tensor does not have shape as intended.")

I_tensor has shape as intended.


#### You can access arbitrary sub-tensors of every tensor

For this, you can use the python's usual slicing notation. For example, to get the second element of each of the deepest lists of `B` in the corresponding tensor, one can simply write

In [19]:
B_tensor[:,:,1]

tensor([[2, 5],
        [0, 1]])

#### Reshaping and Slicing Tensors

**Reshaping Tensors**

Sometimes you need to change the shape of a tensor without changing its data. PyTorch provides the `reshape()` method (or `view()` method) for this purpose. The key requirement is that the total number of elements must remain the same.

**Important points:**
- `tensor.reshape(new_shape)` - returns a tensor with the new shape
- `tensor.view(new_shape)` - similar to reshape, but requires the tensor to be contiguous in memory
- The product of all dimensions in the new shape must equal the total number of elements
- You can use `-1` in one dimension to let PyTorch automatically calculate that dimension's size

**Examples:**
- `tensor.reshape(2, 3)` - reshapes to 2 rows, 3 columns
- `tensor.reshape(-1)` - flattens to 1D
- `tensor.reshape(-1, 2)` - reshapes to 2 columns, with PyTorch calculating the number of rows

**Understanding Tensor Slicing**

PyTorch tensors support Python's slicing notation, which allows you to extract specific parts of tensors. Here are the key concepts:

**Basic Indexing:**
- `tensor[i]` - selects element/slice at index `i` along the first dimension
- `tensor[i, j]` - selects element at position `(i, j)` in a 2D tensor
- `tensor[i, j, k]` - selects element at position `(i, j, k)` in a 3D tensor

**Slicing with Colons (`:`):**
- `tensor[:]` - selects all elements along that dimension
- `tensor[start:end]` - selects elements from `start` (inclusive) to `end` (exclusive)
- `tensor[start:]` - selects all elements from `start` to the end
- `tensor[:end]` - selects all elements from the beginning up to `end` (exclusive)
- `tensor[:, :]` - selects all elements along multiple dimensions

**Step Size:**
- `tensor[start:end:step]` - selects elements with a step size (e.g., `::2` selects every other element)
- `tensor[::2]` - selects every 2nd element (starting from 0)
- `tensor[1::2]` - selects every 2nd element starting from index 1

**Negative Indices:**
- `tensor[-1]` - selects the last element
- `tensor[-2:]` - selects the last two elements

**Combining Dimensions:**
- You can combine these operations across multiple dimensions: `tensor[i, :, j:k]` selects specific slices across different dimensions


**Exercise.** Practice reshaping and slicing tensors:

**Reshaping:** Create a tensor `reshape_tensor` with shape `[2, 3, 4]` filled with consecutive integers starting from 0. Reshape it to: `[6, 4]`, `[24]`, and back to `[2, 3, 4]`.

**Slicing:** Create a tensor `practice_tensor` with shape `[3, 4, 5]` filled with consecutive integers. Then:
1. Extract all elements at index 0 along the first dimension
2. Extract element at index [1, 2] along first two dimensions (all third dimension elements)
3. Extract element at index 2 along the third dimension for all first and second dimension combinations

Print the result and shape for each task.


In [20]:
# Reshaping
# Create a tensor with shape [2, 3, 4] filled with consecutive integers starting from 0
reshape_tensor = torch.arange( 24 ).reshape((2,3,4))  # Use torch.arange and reshape

# Task 1: Reshape to [6, 4]
result_1 = reshape_tensor.reshape((6,4))
print(f"Task 1 shape: {result_1.shape}")

# Task 2: Reshape to [24]
result_2 = result_1.reshape(-1)
print(f"Task 2 shape: {result_2.shape}")

# Task 3: Reshape back to [2, 3, 4]
result_3 = result_2.reshape((2,3,4))
print(f"Task 3 shape: {result_3.shape}")

# Slicing
# Create a tensor with shape [3, 4, 5] filled with consecutive integers starting from 0
practice_tensor = torch.arange( 60 ).reshape((3,4,-1))   # Use torch.arange and reshape

# Task 4: Extract all elements at index 0 along the first dimension
result_4 = practice_tensor[0,:,:]
print(f"Task 4 shape: {result_4.shape}")
print(f"Task 4 result:\n{result_4}\n")

# Task 5: Extract element at index [1, 2] along first two dimensions
result_5 = practice_tensor[1,2,:]
print(f"Task 5 shape: {result_5.shape}")
print(f"Task 5 result:\n{result_5}\n")

# Task 6: Extract element at index 2 along the third dimension for all first and second dimension combinations
result_6 =  practice_tensor[:,:,2]
print(f"Task 6 shape: {result_6.shape}")
print(f"Task 6 result:\n{result_6}\n")





Task 1 shape: torch.Size([6, 4])
Task 2 shape: torch.Size([24])
Task 3 shape: torch.Size([2, 3, 4])
Task 4 shape: torch.Size([4, 5])
Task 4 result:
tensor([[ 0,  1,  2,  3,  4],
        [ 5,  6,  7,  8,  9],
        [10, 11, 12, 13, 14],
        [15, 16, 17, 18, 19]])

Task 5 shape: torch.Size([5])
Task 5 result:
tensor([30, 31, 32, 33, 34])

Task 6 shape: torch.Size([3, 4])
Task 6 result:
tensor([[ 2,  7, 12, 17],
        [22, 27, 32, 37],
        [42, 47, 52, 57]])



#### Tensors can be used in computations element-wise, as long as the dimensions match

In [21]:
B_tensor + B_tensor ** 2 - 0.3 * B_tensor

tensor([[[ 1.7000,  5.4000, 11.1000],
         [18.8000, 28.5000, 40.2000]],

        [[ 0.0000,  0.0000,  0.0000],
         [ 1.7000,  1.7000,  1.7000]]])

**Exercise.** With the help of PyTorch documentation online, find the square root of $B^3$. All operations are applied element-wise

In [22]:
# your solution

torch.sqrt (B_tensor ** 3)

tensor([[[ 1.0000,  2.8284,  5.1962],
         [ 8.0000, 11.1803, 14.6969]],

        [[ 0.0000,  0.0000,  0.0000],
         [ 1.0000,  1.0000,  1.0000]]])

#### Tensors can be either trainable or non-trainable

The trainable tensors are the ones that have `require_gradient` set to `True`.

In [23]:
A_trainable = torch.tensor(A, requires_grad=True)

print(f"A_tensor is trainable: {A_tensor.requires_grad}")
print(f"A_trainable is trainable: {A_trainable.requires_grad}")

A_tensor is trainable: False
A_trainable is trainable: True


## Differentiable Functions and Why They Are So Special

The entire world is now interested in a particular sub-class of functions, called *differentiable functions*.

A differentiable functions is a function $f$ such that $f$ is differentiable with respect to its parameters. Here is an example of a differentiable function $f$ taking $x$ as input and multiplying it by a parameter $p$.

In [24]:
import torch
import torch.nn as nn

In [25]:
p = torch.tensor([ 1.0 ], requires_grad=True)

def f(x):
  global p
  return p * x

Apart from forcing software engineers to dust off their high-school calculus knowledge, what are these differentiable programs actually good for? Why has the entire software engineering and data science world gone crazy over them?

We won't keep you in suspense, here's the "secret":

> Given input-output data, differentiable functions can be taught, through trial and error, to use the right parameters.

So, in the example of `f` above, we could train the function to learn the best value of `p`.

The method enabling this is known as gradient descent training. This has been covered well in many lectures and online resources. If you need a quick refresher, have a look through at the corresponding videos from the Computational Thinking course, available [here](https://disco.ethz.ch/courses/hs24/coti/). The short version is that we can update the parameter `p` using the update:

$p_{new} = p - ∇_pf(x)$

where $∇_p f(x)$ is the gradient of `f` with respect to `p` for the data `x`.

### Gradient Computation in PyTorch
At the helm of gradient-descent training in PyTorch is the `autograd` module. `torch.autograd` is PyTorch's automatic differentiation engine that powers gradient-descent training.

Suppose you take some trainable tensor $x$ and pass it through $f$.

In [26]:
x = torch.tensor([ 5.1 ], requires_grad=True)
output = f(x)
output

tensor([5.1000], grad_fn=<MulBackward0>)

Notice that `output` now has an additional field, `grad_fn`, that was set by the `autograd` system to keep track of what operations have been performed on `x` to arrive at output.

Now, given some expected output for $f(x)$, say $1$, `autograd` allows you to compute an indication of how `p` needs to be changed in order for $f(x)$ to eventually yield the correct outputs. This is done in a process called *backward pass*.

In [27]:
# define the output we expect. I.e. the output of f(x) with the correct p
expected_output = torch.tensor([1.0])
# compute the loss (how "bad" that output is for the current value of p)
loss = (output - expected_output) ** 2
# compute the gradient of f w.r.t. p
loss.backward()

This indication can then by inspected by asking `p` what its gradient is by reading `p.grad`.

In [28]:
p.grad

tensor([41.8200])

This indication can be interpreted as

> Decreasing `p` by some small $\epsilon$ will decrease the loss by $41.82\epsilon$.

And, as you already know, leveraging the negation of gradient as the indication of the direction in which one should modify the parameters in order to descent towards lower values of the loss, gradient descent training is simply the routine under which one iteratively computes and then applies the gradient of the loss function with respect to parameters of the computation to minimise the loss.

**Exercise.** Compute the gradient of the MSE loss with respect to `p` for different values of `p`.

**Your task:** For each value of `p` (0.75, 0.50, 0.25, 0.20, 0.10), complete these steps:
1. Compute the output: `output = f(x)`
2. Compute the MSE loss: `loss = (output - expected_output) ** 2`
3. Call `loss.backward()` to compute gradients
4. Print the gradient: `p.grad.item()`


**After running, observe:** What pattern do you notice in the gradients as `p` changes? What does this tell you about how the gradient behaves when `p` is far from the optimal value?

In [29]:
# For each value of p, complete these steps:
# 1. Compute output = f(x)
# 2. Compute loss = (output - expected_output) ** 2
# 3. Call loss.backward() to compute gradients
# 4. Print the gradient using p.grad.item()

# List of p values to test
p_values = [0.75, 0.50, 0.25, 0.20, 0.10]

# Loop through each value of p
for p_val in p_values:
    # Step 1: Create p tensor (already done - you don't need to change this)
    p = torch.tensor([p_val], requires_grad=True)
    
    # Step 2: TODO - Compute the output using function f
    output = f(x) 
    
    # Step 3: TODO - Compute the MSE loss
    loss = (output - expected_output) ** 2
    
    # Step 4: TODO - Compute gradients by calling backward()
    loss.backward()
    
    # Step 5: Print the gradient (this will work once you complete steps 2-4)
    print(f"For p = {p_val:.2f}, the gradient is {p.grad.item():.4f}")
   

For p = 0.75, the gradient is 28.8150
For p = 0.50, the gradient is 15.8100
For p = 0.25, the gradient is 2.8050
For p = 0.20, the gradient is 0.2040
For p = 0.10, the gradient is -4.9980


### A Basic Training Loop
As hinted on by the above example, modifying the parameters of a differentiable program in the direction opposite to its gradient (i.e. in the direction in which the loss decreases most rapidly) generally guides the differentiable programs towards a minimum in the loss.

This process can be repeated iteratively, to form what is called a *training loop*. A typical training procedure of a differentiable program looks as follows:


1. Initialise the parameters of the differentiable program according to an appropriate scheme.
2. Take the inputs provided and perform a *forward pass* -- apply the program to the inputs.
3. Compute the loss between the expected outputs and the actual outputs of the program.
4. Compute the gradient of the loss with respect to the program's parameters.
5. Scale the gradients by the desired pace of descent -- *the learning rate* -- and update the parameters accordingly.
6. Repeat step 2 to 5 for a number of rounds or until the loss is sufficently small.


You already possess all the basic ingredients necessary to implement such a training procedure yourself. Let's do that.

#### Training loop without PyTorch

To get a feeling for how we can optimize the parameter `p` to some data we will start without the help PyTorch gives. We will still use PyTorch for tensor operations, but this could also be done using NumPy.

In [30]:
def f(x, p):
    return x * p

def grad_f(x, p):
    return x

def loss_function(x, p, y):
    y_hat = f(x, p)
    errors = y_hat - y
    loss = torch.mean(errors**2)
    return loss

def grad_loss_function(x, p, y):
    y_hat = f(x, p)
    grad_y_hat = grad_f(x, p)
    errors = y_hat - y
    grad_errors = grad_y_hat
    grad_loss = torch.mean( 2*errors*grad_errors )
    return grad_loss


def training_loop(x, y, learning_rate: float = 0.1, number_of_iterations: int = 100):
    p = 0.9

    for iteration in range(number_of_iterations):
        # compute the gradient
        grad_loss = grad_loss_function(x, p, y)

        # log the result every 10th iteration
        if iteration%10 == 0:
            print(f"iteration: {iteration:3}, p: {p:5.3f}, gradient: {grad_loss}")

        # update p
        p = p - learning_rate * grad_loss

    return p

In [31]:
p_true = 0.2 # the parameter value of p to be learned
datapoint_count = 10 # the number of datapoints to use for training in every iteration of the training loop

# Generate some artifical data. (datapoint_count, ) gives a one-dimensional array of length datapoint_count.
x = torch.rand((datapoint_count, ), requires_grad=False)
y = x * p_true

p_found = training_loop(x, y, learning_rate=0.1)
print(f"The true value is {p_true}, the value learned by gradient descent is {p_found}")

iteration:   0, p: 0.900, gradient: 0.4012792706489563
iteration:  10, p: 0.588, gradient: 0.2223636656999588
iteration:  20, p: 0.415, gradient: 0.12321992963552475
iteration:  30, p: 0.319, gradient: 0.06828070431947708
iteration:  40, p: 0.266, gradient: 0.037836864590644836
iteration:  50, p: 0.237, gradient: 0.020966818556189537
iteration:  60, p: 0.220, gradient: 0.011618485674262047
iteration:  70, p: 0.211, gradient: 0.006438233889639378
iteration:  80, p: 0.206, gradient: 0.0035676597617566586
iteration:  90, p: 0.203, gradient: 0.0019769687205553055
The true value is 0.2, the value learned by gradient descent is 0.20191103219985962


**Exercise.** Play around with the number of iterations and learning rate. What happens if the learning rate is very high (>5) or very small (<1e-6)?


#### Training loop with PyTorch

We need to define the function using PyTorch's syntax.
A function in PyTorch is a `class` with a `forward` function.
Forward defines what should happen when you say `f(x)` and the building blocks
are the functions in `torch.nn`, e.g., `nn.Linear`. The next section will look at this in details

In [32]:

class f_function(nn.Module):
    def __init__(self):
        super().__init__()
        # Make a linear function (y=ax+b) from input to output.
        # bias=False means b=0, so we get y = p*x
        self.p = nn.Linear(1, 1, bias=False)

        # set p to a fixed value
        # the torch.no_grad() removes gradient computations, so p.grad does not change
        with torch.no_grad():
            self.p.weight[0][0] = 0.9

    def forward(self, x):
        return self.p(x)

def train_f(x, y, learning_rate: float = 0.1, number_of_iterations: int = 100):
    # define the function
    f = f_function()
    # define the loss function as the mean squared loss
    loss_fn = nn.MSELoss()
    # define the optimizer as classic gradient decent
    optimizer = torch.optim.SGD(f.parameters(), lr=learning_rate)

    for iteration in range(number_of_iterations):
        # perform a "forward pass" (apply f to x)
        y_hat = f(x)

        # compute the loss
        loss = loss_fn(y_hat, y)

        # zero the gradients computed in the previous step
        optimizer.zero_grad()

        # calculate the gradients of the parameters of the net
        loss.backward()

        # use the gradients to update the weights of the network
        optimizer.step()

    # return p
    return f.p.weight[0][0]

**Exercise.** Complete the code below and compare the results to the implementation without PyTorch

In [33]:
# pytorch needs the shape to be an array of shape (datapoint_count, 1)
x_pytorch = x.reshape( (datapoint_count,1) )
y_pytorch = y.reshape( (datapoint_count,1))

p_found = train_f(x_pytorch, y_pytorch, learning_rate=0.1)
print(f"The true value is {p_true}, the value learned by gradient descent is {p_found}")

The true value is 0.2, the value learned by gradient descent is 0.20191101729869843


## Introducing Neural Networks
Neural networks are a particular class of differentiable programs, for which it has been theoretically proven that they can learn to approximate an arbitrary integrable function arbitrarily well, as long as they are given enough *representational power*.

*Here is where the deep learning black magic begins.*

Classical programs consist of a sequence of specific operations such as addition or conditional value assignment. Neural networks are differentiable programs that consist of a sequence of amenable elementary building blocks, traditionally referred to as *layers*, that can ultimately perform a wide variety of operations. The "bigger" these layers are, the more complex behaviour they can learn to exhibit.

There exists several popular types of neural network building blocks, including the trainable *linear layer*, or the non-trainable *activation*, *softmax*, and *dropout layers*, to name but a few. The combination of a linear layer and an activation layer is sometimes referred to as *dense layer* and is the basic building block of a *deep neural network*.

The amount of *representational power* network has is determined by the sizes of its trainable layers. Linear layers have a "width" (the number of constituent neurons). The wider the layer, the more fine-grained operation it is capable of representing. Whether it can learn to represent this operation is, however, an entirely different question.

In [34]:
import torch
import torch.nn as nn

### Constructing Neural Networks
Without further ado, let use these building blocks to form neural networks.

Knowing the format of the operation of individual layers, you could go ahead and implement them manually. To avoid uncanny detail, we will instead use the ready-made implementations of these layers from the `torch.nn` module.

In general, the [documentation](https://pytorch.org/docs/stable/nn.html) of the `torch.nn` module is what you want to turn to to understand a new layer type.

We walk you through creating instances of various layer types in the code below. We directly use the instances to operate input data.

#### Linear Layers

Let us begin with the most basic layer in deep learning, the Linear layer.

In [35]:
# construct a linear layer that takes a tensor of size (3,) and produces a
#  tensor of size (5,)
linear_layer = torch.nn.Linear(3, 5)

# pass [1, 2, 3] through the layer
example_input = torch.tensor([ 1, 2, 3 ], dtype=torch.float)
linear_layer(example_input)

tensor([-2.2916, -0.6904,  0.6860,  1.1652,  0.5374], grad_fn=<ViewBackward0>)

Notice that as promised in the call to `nn.Linear(3, 5)`, the output tensor has 5 entries. Its output values are the result of an internal state (the layer *weights*) that has been initialised with random weights.

#### Activation Layers

Several types of activation layers exist, most notably the logistic sigmoid, rectified linear unit (ReLU), and the hyperbolic tangent. Each of these has a layer in `torch.nn`.

In [36]:
# construct a ReLU layer
relu_layer = torch.nn.ReLU()

# pass [ -3, -2, -1, 0, +1, +2, +3 ] through the ReLU layer
example_input = torch.tensor([ -3, -2, -1, 0, +1, +2, +3 ], dtype=torch.float)
relu_layer(example_input)

tensor([0., 0., 0., 0., 1., 2., 3.])

In [37]:
# construct a sigmoid layer
sigmoid_layer = torch.nn.Sigmoid()

# pass [ -3, -2, -1, 0, +1, +2, +3 ] through the sigmoid layer
sigmoid_layer(example_input)

tensor([0.0474, 0.1192, 0.2689, 0.5000, 0.7311, 0.8808, 0.9526])

In [38]:
# construct a tanh layer
tanh_layer = torch.nn.Tanh()

# pass [ -3, -2, -1, 0, +1, +2, +3 ] through the tanh layer
tanh_layer(example_input)

tensor([-0.9951, -0.9640, -0.7616,  0.0000,  0.7616,  0.9640,  0.9951])

As you can see, going from minus infinity towards infinity around 0, the ReLU transits from constant 0 to linear behaviour at 0, the logistic sigmoid proceeds to climb from 0 towards 1, and tanh climbs from -1 towards +1.

#### Dropout Layers

It is sometimes to the advantage of model training to "drop out" some of the incoming values at random. To this end, `torch.nn` provides the `Dropout` layer, which can be parameterised at construction with the probability of an input value being dropped out.

In [44]:
# construct a dropout layer with probability 0.5
dropout_layer = torch.nn.Dropout(p=0.5)

# pass [ -3, -2, -1, 0, +1, +2, +3 ] through the dropout layer
dropout_layer(example_input)

# with p=0.5, roughly half of the inputs should be dropped out on average,
#  and the remaining outputs are scaled up by 1/(1-p) == 2
# run this snippet multiple times to observe the effects of random dropout

tensor([-0., -4., -2.,  0.,  2.,  0.,  6.])

#### Softmax
Sometimes we wish to interpret an $n$-dimensional vector of real values as scores in favour of a single one of $n$ discrete elements possessing a certain property. To this end, we often use the "softmax" layer.

The softmax layer takes the $n$-dimensional vector of real values and produces an $n$-dimensional vector of values between $0$ and $1$, whose individual entries sum up to $1$.

The bigger an entry of the input vector is relative to other entries, the closer its corresponding value in the output vector is to $1$.

In [45]:
# construct a softmax layer
softmax_layer = torch.nn.Softmax(dim=0)

# pass [ -3, -2, -1, 0, +1, +2, +3 ] through the softmax layer
softmax_layer(example_input)

tensor([0.0016, 0.0043, 0.0116, 0.0315, 0.0856, 0.2328, 0.6327])

In [ ]:
# pass [ -1, 2, 5, 100 ] through the softmax layer
softmax_layer(torch.tensor([ -1, 2, 5, 100 ], dtype=torch.float))

### Putting the Layers Together -- Implementing a DNN


Neural networks are graphs of layers. In PyTorch, we generally tend to implement our neural networks as classes whose constructors construct the constituent parts of the network, and whose `forward` function passes the data through these parts.

Below is an example implementation of a shallow neural network. This network is *shallow* as it contains only one hidden trainable layer (=layer that is not an input or output layer).


In [46]:
class ShallowNeuralNet(nn.Module):
    def __init__(self, input_width: int, hidden_layer_width: int, output_width):
        super().__init__()
        self.hidden_layer = nn.Linear(input_width, hidden_layer_width)
        self.hidden_relu = nn.ReLU()
        self.output_layer = nn.Linear(hidden_layer_width, output_width)

    def forward(self, input):
        hidden_trainable_output = self.hidden_layer(input)
        hidden_relu_output = self.hidden_relu(hidden_trainable_output)
        output = self.output_layer(hidden_relu_output)
        return output

Once the network behaviour has been described in this fashion, we can create an instance of the entire network at once and use it to process data in exactly the same way as we would use layers.

In [47]:
shallow_nn_instance = ShallowNeuralNet(5, 10, 2)

example_input = torch.ones(5)
shallow_nn_instance(example_input)

tensor([ 0.1580, -0.0544], grad_fn=<ViewBackward0>)

**Exercise.** Complete the forward pass below to arrive at an implementation of a deep ReLU neural networks with layer profile given by a list of integers.

In [48]:
class DeepNeuralNet(nn.Module):
  def __init__(self, input_width, hidden_layer_profile, output_width, output_activation=None):
    super().__init__()
    self.layers = nn.ModuleList()

    # create the first hidden layer
    self.layers.append(nn.Linear(input_width, hidden_layer_profile[0]))
    self.layers.append(nn.ReLU())

    # create the internal hidden layers
    for in_width, out_width in zip(hidden_layer_profile[0:-1], hidden_layer_profile[1:]):
      self.layers.append(nn.Linear(in_width, out_width))
      self.layers.append(nn.ReLU())

    self.layers = nn.Sequential(*self.layers)

    # create the output layer
    self.output_layer = nn.Linear(hidden_layer_profile[-1], output_width)
    self.output_activation = nn.Identity() if not output_activation else output_activation

  def forward(self, input):
    x = input

    # loop through the layers to produce the output of the hidden network
    for layer in self.layers:
      # TODO: pass the intermediate output of the previous layer through the current layer
      x = layer(x)

    # TODO: produce the output of the network from the intermediate output of the last hidden layer
    output_before_activation = self.output_layer(x)

    # TODO: engage the optional activation in self.output_activation on the output_before_activation
    output = self.output_activation(output_before_activation)

    return output

**Exercise.** Test the class for generic deep neural networks below. Does everything work as expected?


In [49]:
# try passing a tensor of random numbers through the network
input1 = torch.rand((10,))
deep_nn_instance = DeepNeuralNet(input_width=10, hidden_layer_profile=[10, 7, 5], output_width=1)

deep_nn_instance(input1)

tensor([-0.0523], grad_fn=<ViewBackward0>)

### A Training Loop for Neural Networks
In a previous section concerning differentiable functions, we introduced the intuition for using the gradient information due to a choice of loss function to find optimal parameters of a differentiable function.

This is exactly what we do for neural networks as well in order to train them to have the behaviour we desire of them.

We give code for optimisation of a neural network `net` with particular loss function `loss` on dataset loaded by a `dataloader` below, and comment on it step by step.

In [ ]:
# Use the gpu if you are connected to a runtime with one.
# This is highly recommended, as it makes computations much faster
# (especially later on for the larger models)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def training_loop(
        dataloader: torch.utils.data.DataLoader,
        net: nn.Module,
        loss_fn: nn.Module,
        optimiser: torch.optim.Optimizer,
        verbosity: int=3,
        device = DEVICE):
    size = len(dataloader.dataset)
    last_print_point = 0
    current = 0

    acc_loss = 0
    acc_count = 0
    net.train()
    # for every slice (X, y) of the training dataset
    for batch, (X, y) in enumerate(dataloader):
        X = X.to(device) # torch.Tensor
        y = y.to(device)

        # perform a forward pass to compute the outputs of the net
        pred = net(X)

        # calculate the loss between the outputs of the net and the desired outputs
        loss_val = loss_fn(pred, y)
        acc_loss += loss_val.item()
        acc_count += 1

        # zero the gradients computed in the previous step
        optimiser.zero_grad()

        # calculate the gradients of the parameters of the net
        loss_val.backward()

        # use the gradients to update the weights of the network
        optimiser.step()

        # compute how many datapoints have already been used for training
        current = batch * len(X)

        # report on the training progress roughly every 10% of the progress
        if verbosity >= 3 and (current - last_print_point) / size >= 0.1:
            loss_val = loss_val.item()
            last_print_point = current
            print(f"loss: {loss_val:>7f}  [{current:>5d}/{size:>5d}]")
    return acc_loss / acc_count

We now possess all the tools necessary for constructing simple neural networks and for training them towards some particular behaviour by gradient descent loss minimisation. Let us put these tools to good use.

### Learning a Boolean Function



Let us consider the problem of learning a random Boolean function $f: \left\{ 0,1 \right\}^n \to \left\{ 0,1 \right\}^n$. It might sound a bit boring at first, but bear with us, as it is a very natural and tractable example for the examination of the representational power of various types of neural networks.

In [66]:
def make_binary_array(number: int, length: int) -> list:
	return [ (number>>k)&1 for k in range(0, length) ]

data_x_list = []

for i in range(0, 256):
  data_x_list.append(make_binary_array(i, 8))

data_x = torch.tensor(data_x_list, dtype=torch.float)
data_y = torch.randint(low=0, high=2, size=(256, 8)).type(torch.float)

# Move data to compute device
data_x = data_x.to(DEVICE)
data_y = data_y.to(DEVICE)

dataset = torch.utils.data.TensorDataset(data_x, data_y)

The above code generates a dataset of $(x, y)$ pairs where $x$ is any $n=8$-bit signal and $y = f(x)$, with $f$ chosen uniformly at random.

Given these examples, can we learn a neural network that performs the function of $f$? Yes! Just run the code below

In [ ]:
def testing_loop(dataloader, net : nn.Module, device = DEVICE):
  size = len(dataloader.dataset)
  last_print_point = 0
  current = 0

  acc_correct = 0
  acc_count = 0

  net.eval()
  # for every slice (X, y) of the training dataset
  with torch.no_grad():
    for batch, (X, y) in enumerate(dataloader):
        X = X.to(device)
        y = y.to(device)

        # perform a forward pass to compute the outputs of the net
        pred = net(X)

        # round the predictions (0 - 0.5 towards zero, >0.5 towards one)
        pred_rounded = torch.round(pred)

        # compute the number of correct entries
        acc_correct += torch.count_nonzero(pred_rounded == y).item()
        acc_count += y.numel()


  return acc_correct / acc_count

In [68]:
from tqdm import tqdm

def train(dataloader, net, loss_fn, optimiser, epochs, epoch_frequency=50, device=DEVICE, verbosity=3):
  least_loss = None
  if verbosity < 2:
    for t in tqdm(range(epochs)):
      mean_loss = training_loop(dataloader, net, loss_fn, optimiser, verbosity=verbosity)
      accuracy = testing_loop(dataloader, net)
      if not least_loss or mean_loss < least_loss:
        least_loss = mean_loss
  else:
    for t in range(epochs):
        if verbosity >= 3:
          print(f"Epoch {t+1}\n-------------------------------")

        mean_loss = training_loop(dataloader, net, loss_fn, optimiser, verbosity=verbosity)
        accuracy = testing_loop(dataloader, net)
        if not least_loss or mean_loss < least_loss:
          least_loss = mean_loss

        if verbosity >= 2 and t%epoch_frequency == 0:
          print(f"Epoch {t:4}: mean loss {mean_loss:.5f}, validation accuracy {accuracy:7.2%}")
        if verbosity >= 3:
          print("\n")

  if verbosity >= 1:
    print(f"\nTraining complete, least loss {least_loss}, final validation accuracy {accuracy:.2%}")

  return least_loss

In [71]:
net = DeepNeuralNet(input_width=8, hidden_layer_profile=[256], output_width=8, output_activation=nn.Sigmoid()).to(DEVICE)

training_dataloader = torch.utils.data.DataLoader(dataset, batch_size=32, shuffle=True, drop_last=True)
loss_fn = nn.BCELoss()
optimiser = torch.optim.Adam(net.parameters(), lr=2e-2)

least_loss = train(training_dataloader, net, loss_fn, optimiser, epochs=400, verbosity=1)


  3%|▎         | 12/400 [00:00<00:03, 117.40it/s]

0
torch.Size([32, 8])
<class 'torch.Tensor'>
1
torch.Size([32, 8])
<class 'torch.Tensor'>
2
torch.Size([32, 8])
<class 'torch.Tensor'>
3
torch.Size([32, 8])
<class 'torch.Tensor'>
4
torch.Size([32, 8])
<class 'torch.Tensor'>
5
torch.Size([32, 8])
<class 'torch.Tensor'>
6
torch.Size([32, 8])
<class 'torch.Tensor'>
7
torch.Size([32, 8])
<class 'torch.Tensor'>
0
torch.Size([32, 8])
<class 'torch.Tensor'>
1
torch.Size([32, 8])
<class 'torch.Tensor'>
2
torch.Size([32, 8])
<class 'torch.Tensor'>
3
torch.Size([32, 8])
<class 'torch.Tensor'>
4
torch.Size([32, 8])
<class 'torch.Tensor'>
5
torch.Size([32, 8])
<class 'torch.Tensor'>
6
torch.Size([32, 8])
<class 'torch.Tensor'>
7
torch.Size([32, 8])
<class 'torch.Tensor'>
0
torch.Size([32, 8])
<class 'torch.Tensor'>
1
torch.Size([32, 8])
<class 'torch.Tensor'>
2
torch.Size([32, 8])
<class 'torch.Tensor'>
3
torch.Size([32, 8])
<class 'torch.Tensor'>
4
torch.Size([32, 8])
<class 'torch.Tensor'>
5
torch.Size([32, 8])
<class 'torch.Tensor'>
6
torch.Si

  9%|▉         | 36/400 [00:00<00:03, 114.27it/s]

0
torch.Size([32, 8])
<class 'torch.Tensor'>
1
torch.Size([32, 8])
<class 'torch.Tensor'>
2
torch.Size([32, 8])
<class 'torch.Tensor'>
3
torch.Size([32, 8])
<class 'torch.Tensor'>
4
torch.Size([32, 8])
<class 'torch.Tensor'>
5
torch.Size([32, 8])
<class 'torch.Tensor'>
6
torch.Size([32, 8])
<class 'torch.Tensor'>
7
torch.Size([32, 8])
<class 'torch.Tensor'>
0
torch.Size([32, 8])
<class 'torch.Tensor'>
1
torch.Size([32, 8])
<class 'torch.Tensor'>
2
torch.Size([32, 8])
<class 'torch.Tensor'>
3
torch.Size([32, 8])
<class 'torch.Tensor'>
4
torch.Size([32, 8])
<class 'torch.Tensor'>
5
torch.Size([32, 8])
<class 'torch.Tensor'>
6
torch.Size([32, 8])
<class 'torch.Tensor'>
7
torch.Size([32, 8])
<class 'torch.Tensor'>
0
torch.Size([32, 8])
<class 'torch.Tensor'>
1
torch.Size([32, 8])
<class 'torch.Tensor'>
2
torch.Size([32, 8])
<class 'torch.Tensor'>
3
torch.Size([32, 8])
<class 'torch.Tensor'>
4
torch.Size([32, 8])
<class 'torch.Tensor'>
5
torch.Size([32, 8])
<class 'torch.Tensor'>
6
torch.Si

 16%|█▌        | 62/400 [00:00<00:02, 122.61it/s]

...

5
torch.Size([32, 8])
<class 'torch.Tensor'>
6
torch.Size([32, 8])
<class 'torch.Tensor'>
7
torch.Size([32, 8])
<class 'torch.Tensor'>
0
torch.Size([32, 8])
<class 'torch.Tensor'>
1
torch.Size([32, 8])
<class 'torch.Tensor'>
2
torch.Size([32, 8])
<class 'torch.Tensor'>
3
torch.Size([32, 8])
<class 'torch.Tensor'>
4
torch.Size([32, 8])
<class 'torch.Tensor'>
5
torch.Size([32, 8])
<class 'torch.Tensor'>
6
torch.Size([32, 8])
<class 'torch.Tensor'>
7
torch.Size([32, 8])
<class 'torch.Tensor'>
0
torch.Size([32, 8])
<class 'torch.Tensor'>
1
torch.Size([32, 8])
<class 'torch.Tensor'>
2
torch.Size([32, 8])
<class 'torch.Tensor'>
3
torch.Size([32, 8])
<class 'torch.Tensor'>
4
torch.Size([32, 8])
<class 'torch.Tensor'>
5
torch.Size([32, 8])
<class 'torch.Tensor'>
6
torch.Size([32, 8])
<class 'torch.Tensor'>
7
torch.Size([32, 8])
<class 'torch.Tensor'>
0
torch.Size([32, 8])
<class 'torch.Tensor'>
1
torch.Size([32, 8])
<class 'torch.Tensor'>
2
torch.Size([32, 8])
<class 'torch.Tensor'>
3
torch.Si

 97%|█████████▋| 389/400 [00:03<00:00, 119.32it/s]

4
torch.Size([32, 8])
<class 'torch.Tensor'>
5
torch.Size([32, 8])
<class 'torch.Tensor'>
6
torch.Size([32, 8])
<class 'torch.Tensor'>
7
torch.Size([32, 8])
<class 'torch.Tensor'>
0
torch.Size([32, 8])
<class 'torch.Tensor'>
1
torch.Size([32, 8])
<class 'torch.Tensor'>
2
torch.Size([32, 8])
<class 'torch.Tensor'>
3
torch.Size([32, 8])
<class 'torch.Tensor'>
4
torch.Size([32, 8])
<class 'torch.Tensor'>
5
torch.Size([32, 8])
<class 'torch.Tensor'>
6
torch.Size([32, 8])
<class 'torch.Tensor'>
7
torch.Size([32, 8])
<class 'torch.Tensor'>
0
torch.Size([32, 8])
<class 'torch.Tensor'>
1
torch.Size([32, 8])
<class 'torch.Tensor'>
2
torch.Size([32, 8])
<class 'torch.Tensor'>
3
torch.Size([32, 8])
<class 'torch.Tensor'>
4
torch.Size([32, 8])
<class 'torch.Tensor'>
5
torch.Size([32, 8])
<class 'torch.Tensor'>
6
torch.Size([32, 8])
<class 'torch.Tensor'>
7
torch.Size([32, 8])
<class 'torch.Tensor'>
0
torch.Size([32, 8])
<class 'torch.Tensor'>
1
torch.Size([32, 8])
<class 'torch.Tensor'>
2
torch.Si

100%|██████████| 400/400 [00:03<00:00, 122.76it/s]

1
torch.Size([32, 8])
<class 'torch.Tensor'>
2
torch.Size([32, 8])
<class 'torch.Tensor'>
3
torch.Size([32, 8])
<class 'torch.Tensor'>
4
torch.Size([32, 8])
<class 'torch.Tensor'>
5
torch.Size([32, 8])
<class 'torch.Tensor'>
6
torch.Size([32, 8])
<class 'torch.Tensor'>
7
torch.Size([32, 8])
<class 'torch.Tensor'>
0
torch.Size([32, 8])
<class 'torch.Tensor'>
1
torch.Size([32, 8])
<class 'torch.Tensor'>
2
torch.Size([32, 8])
<class 'torch.Tensor'>
3
torch.Size([32, 8])
<class 'torch.Tensor'>
4
torch.Size([32, 8])
<class 'torch.Tensor'>
5
torch.Size([32, 8])
<class 'torch.Tensor'>
6
torch.Size([32, 8])
<class 'torch.Tensor'>
7
torch.Size([32, 8])
<class 'torch.Tensor'>
0
torch.Size([32, 8])
<class 'torch.Tensor'>
1
torch.Size([32, 8])
<class 'torch.Tensor'>
2
torch.Size([32, 8])
<class 'torch.Tensor'>
3
torch.Size([32, 8])
<class 'torch.Tensor'>
4
torch.Size([32, 8])
<class 'torch.Tensor'>
5
torch.Size([32, 8])
<class 'torch.Tensor'>
6
torch.Size([32, 8])
<class 'torch.Tensor'>
7
torch.Si

Okay, this is encouraging. We are getting a loss in the order of $10^{-3}$ (which is relatively little for mean binary cross-entropy) and 100% accuracy. You will notice that as the training progresses, the loss tends to decrease and the accuracy increases. You will also notice that our neural network has only one hidden layer of 256 neurons. But are all those neurons really necessary?

Let's push things to an extremum and consider a network that has exactly one neuron in its hidden layer. In other words, all of the information about the input the output neurons have must be contained in exactly one activated number, and the hidden layer of such a network is an information bottleneck. All other parameters constant, what sort of loss values and accuracies will we be getting under such circumstances?

In [ ]:
slender_net = DeepNeuralNet(input_width=8, hidden_layer_profile=[ 1 ], output_width=8, output_activation=nn.Sigmoid()).to(DEVICE)

training_dataloader = torch.utils.data.DataLoader(dataset, batch_size=32, shuffle=True)
loss_fn = nn.BCELoss()
optimiser = torch.optim.Adam(slender_net.parameters(), lr=2e-2)

least_slender_loss = train(training_dataloader, slender_net, loss_fn, optimiser, epochs=400, verbosity=1)

We see that with one hidden neuron, we learn to predict the values of $f(x)$ only marginally better than a coin flip, and that this corresponds to a relatively large value of the binary cross-entropy loss.

Okay. So there is a number of hidden neurons (256) that is sufficient for learning $f$ with 100% accuracy, and there is a number of hidden neurons (1) that is clearly insufficient to learn anything but some rough indication of the correct output.

*   With one hidden neuron, we have starved the network of representational power to the extent that it is only slightly better than tossing a fair coin at predicting $f$.
*   With 256 hidden neurons, we have given the network enough representational power to learn $f$. Perhaps even too much.

What happens in between these two extrema? And, is there a number of neurons beyond which the network fails to learn $f$ correctly but for which $f$ can still be learned?

**Exercise.** Find the least number of neurons $w_1$ such that the training of a shallow neural network with $w$ hidden neurons can still learn to execute $f$ with loss of at most $0.001$. Remember that you can adjust the learning rate and the number of epochs to get finer and more resource-efficient training. You can also set `verbosity=1` to avoid long listings of losses, though verbosity of above `1` might help with the investigation of whether the training losses plateau out.

You can also play with the number of epochs and learning rate to ensure the model has finished training.

In [ ]:
w_1 = None
net = DeepNeuralNet(input_width=8, hidden_layer_profile=[ w_1 ], output_width=8, output_activation=nn.Sigmoid()).to(DEVICE)

training_dataloader = torch.utils.data.DataLoader(dataset, batch_size=32, shuffle=True)
loss_fn = nn.BCELoss()
optimiser = torch.optim.Adam(net.parameters(), lr=2e-2)
least_loss = train(training_dataloader, net, loss_fn, optimiser, epochs=400, verbosity=1)

**Exercise.** It has been theoretically proven that deep neural networks (that is, networks with more than one hidden layer) can learn the same functions as shallow neural nets while using comparatively fewer neurons and trainable weights. Can you find a `w_2`, being a minimal number of neurons sufficient to learn the function $f$ with loss of at most $0.001$, such that the `w_2` neurons can be distributed in multiple hidden layers? The number of layers you end up using is up to you.

Hint: Decrease the learning rate and increase the number of epochs

In [ ]:
# for example, distribute over three layers
w_2_1 = None
w_2_2 = None
w_2_3 = None

w_2 = w_2_1 + w_2_2 + w_2_3
print(f"Using {w_2} neurons")
net = DeepNeuralNet(input_width=8, hidden_layer_profile=[ w_2_1, w_2_2, w_2_3, ], output_width=8, output_activation=nn.Sigmoid()).to(DEVICE)

training_dataloader = torch.utils.data.DataLoader(dataset, batch_size=32, shuffle=True)
loss_fn = nn.BCELoss()
optimiser = torch.optim.Adam(net.parameters(), lr=2e-2)
least_loss = train(training_dataloader, net, loss_fn, optimiser, epochs=400, verbosity=2)

### Section Takeaway

We have seen that neural networks can be constructed as directed graphs of more elementary building blocks (shallow and deep nets).

We have also introduced the training loop for a neural network that uses much of PyTorch machinery to perform gradient descent.

We have used the above to train networks that learn a Boolean function. We observed that not all networks can learn all functions, and that the number of neurons and their arrangement (or, more precisely, trainable weights and their role within the network) influence the ability of a network to learn to approximate a function. Experimenting, we got the intuitive feel of the notion of *representational power*.

Using all that has been learned, we can now go and train networks that are perhaps more suitable for real-world applications.

## Bias-Variance Tradeoff

In the previous section, we explored how the representational power of a neural network affects its ability to learn functions. We observed that networks with too few neurons struggle to learn, while networks with many neurons can learn complex functions. However, there's a fundamental tradeoff in machine learning that we haven't explicitly discussed yet: the **bias-variance tradeoff**.

The bias-variance tradeoff is a central concept in machine learning that helps us understand the relationship between model complexity and generalization error. The total error of a model can be decomposed into three components:

1. **Bias**: The error due to overly simplistic assumptions in the learning algorithm. High bias can cause a model to miss relevant relations between features and target outputs (underfitting).

2. **Variance**: The error due to sensitivity to small fluctuations in the training set. High variance can cause a model to model the random noise in the training data rather than the intended outputs (overfitting).

3. **Irreducible Error**: The noise inherent in the problem itself, which cannot be reduced by any model.

The key insight is that as model complexity increases:
- **Bias decreases** (the model can fit more complex patterns)
- **Variance increases** (the model becomes more sensitive to training data noise)
- There's an optimal complexity that minimizes the total error

Let's explore this concept with a simple 1D polynomial regression example.


### Exploring Bias-Variance Tradeoff

We'll use polynomial regression to explore the bias-variance tradeoff. The true underlying function is a polynomial, and we'll add noise to simulate real-world data. 


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

# Set random seed for reproducibility
torch.manual_seed(1)
np.random.seed(1)

# Define the true underlying function (a polynomial of degree 2)
def true_function(x):
    """True function: 2nd degree polynomial"""
    return x**2

# Generate data points for visualization
x_true = np.linspace(-1, 1, 1000)
y_true = true_function(x_true)

# Generate data with noise and split into train/validation/test
n_total = 15
x_all = np.random.uniform(-1, 1, n_total)
noise = np.random.normal(0, 0.1, n_total)
y_all = true_function(x_all) + noise

# Split into train, validation, test
n_train = 5
n_val = 5
n_test = 5

indices = np.random.permutation(n_total)
train_idx = indices[:n_train]
val_idx = indices[n_train:n_train+n_val]
test_idx = indices[n_train+n_val:]

x_train = x_all[train_idx]
y_train = y_all[train_idx]
x_val = x_all[val_idx]
y_val = y_all[val_idx]
x_test = x_all[test_idx]
y_test = y_all[test_idx]

# Plot the true function and data splits
plt.figure(figsize=(10, 6))
plt.plot(x_true, y_true, 'b-', label='True function', linewidth=2)
plt.scatter(x_train, y_train, color='red', s=50, alpha=0.6, label='Training data', zorder=5)
plt.scatter(x_val, y_val, color='orange', s=50, alpha=0.6, label='Validation data', zorder=5)
plt.scatter(x_test, y_test, color='green', s=50, alpha=0.6, label='Test data', zorder=5)
plt.xlabel('x', fontsize=12)
plt.ylabel('y', fontsize=12)
plt.title('True Function (2nd degree) and Data Splits', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


Run the code below to fit polynomials of different degrees. Observe how model complexity affects the fit.


In [ ]:
x_mean = x_train.mean()
x_std = x_train.std()

def normalize(x):
    return (x - x_mean) / x_std

x_train_n = normalize(x_train)
x_val_n   = normalize(x_val)
x_test_n  = normalize(x_test)
x_true_n  = normalize(x_true)


class PolynomialModel(nn.Module):
    def __init__(self, degree):
        super().__init__()
        self.degree = degree
        self.linear = nn.Linear(degree + 1, 1, bias=True)

    def forward(self, x):
        x_squeezed = x.squeeze(-1)
        features = torch.stack(
            [x_squeezed ** i for i in range(self.degree + 1)],
            dim=1
        )
        return self.linear(features)


def compute_error(model, x_tensor, y_tensor, device):
    model.eval()
    with torch.no_grad():
        pred = model(x_tensor.to(device)).squeeze(-1)
        loss_fn = nn.MSELoss()
        error = loss_fn(pred, y_tensor.to(device))
    return error.item()


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

x_train_tensor = torch.FloatTensor(x_train_n).unsqueeze(1).to(device)
y_train_tensor = torch.FloatTensor(y_train).to(device)
x_val_tensor   = torch.FloatTensor(x_val_n).unsqueeze(1).to(device)
y_val_tensor   = torch.FloatTensor(y_val).to(device)
x_test_tensor  = torch.FloatTensor(x_test_n).unsqueeze(1).to(device)
y_test_tensor  = torch.FloatTensor(y_test).to(device)
x_true_tensor  = torch.FloatTensor(x_true_n).unsqueeze(1).to(device)


# Train models for different polynomial degrees
degrees = [1, 2, 3, 4]
models = {}
predictions = {}
train_errors = {}
val_errors = {}
test_errors = {}

for degree in degrees:
    # Create and train model
    model = PolynomialModel(degree).to(device)
    loss_fn = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.01)
    
    train_dataset = TensorDataset(x_train_tensor, y_train_tensor.unsqueeze(1))
    train_dataloader = DataLoader(train_dataset, batch_size=len(train_dataset), shuffle=False)
    
    train(train_dataloader, model, loss_fn, optimizer, epochs=2000, verbosity=0)
    
    # Compute errors on all sets
    train_errors[degree] = compute_error(model, x_train_tensor, y_train_tensor, device)
    val_errors[degree] = compute_error(model, x_val_tensor, y_val_tensor, device)
    test_errors[degree] = compute_error(model, x_test_tensor, y_test_tensor, device)
    
    # Get predictions for visualization
    model.eval()
    with torch.no_grad():
        predictions[degree] = model(x_true_tensor).squeeze(-1).cpu().numpy()
    
    models[degree] = model
    print(f"Degree {degree}: Train Error = {train_errors[degree]:.4f}, Val Error = {val_errors[degree]:.4f}")


# Visualize the polynomial fits
fig, axes = plt.subplots(1, len(degrees), figsize=(20, 6))
fig.suptitle('Polynomial Fits of Different Degrees', fontsize=16, y=1.02)
axes = axes.flatten()

for idx, degree in enumerate(degrees):
    ax = axes[idx]
    ax.plot(x_true, y_true, 'b-', label='True function', linewidth=2, alpha=0.7)
    ax.scatter(x_train, y_train, color='red', s=30, alpha=0.6, label='Training data', zorder=5)
    ax.plot(x_true, predictions[degree], 'g--', label=f'Degree {degree} fit', linewidth=2)
    ax.set_title(f'Degree {degree}\nTrain: {train_errors[degree]:.3f}, Val: {val_errors[degree]:.3f}', fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)
    ax.set_ylim([-0.5, 2])

plt.tight_layout()
plt.show()

# Plot train and validation errors
plt.figure(figsize=(10, 6))
plt.plot(degrees, [train_errors[d] for d in degrees], 'o-', label='Train Error', linewidth=2, markersize=8)
plt.plot(degrees, [val_errors[d] for d in degrees], 's-', label='Validation Error', linewidth=2, markersize=8)
plt.xlabel('Polynomial Degree', fontsize=12)
plt.ylabel('MSE Error', fontsize=12)
plt.title('Train vs Validation Error for Different Polynomial Degrees', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.xticks(degrees)
plt.tight_layout()
plt.show()


**Discussion with your neighbor:** What do you observe about the different polynomial fits? Which degrees seem to underfit? Which seem to overfit?

**Exercise.** Write down your observations in the cell below.


In [ ]:
# Write your observations here


Based on the validation error, we can see which model performs best. However, the true test of a model's performance is how well it generalizes to completely unseen data.

In [ ]:
# Enter the degree with the lowest validation error
best_degree = None
print(f"Best model test error = {test_errors[best_degree]:.4f}")


### Interesting Outlook: Double Descent

Wait, there's a twist! The bias-variance tradeoff we just explored isn't the whole story. Recent research has discovered something surprising: **double descent**.

**The Plot Twist:** As model complexity increases beyond the point where it can perfectly fit the training data, the test error doesn't keep getting worse—it actually **starts improving again**! This creates a second descent in the error curve, challenging the classical understanding that more complex models always overfit.

**Why this matters:** This phenomenon has been observed in deep neural networks, random forests, and other modern ML models. It suggests that in the overparameterized regime (more parameters than training samples), the relationship between complexity and generalization is more nuanced than we thought.

**Want to learn more?**
- [Video: Understanding Double Descent](https://www.youtube.com/watch?v=z64a7USuGX0)
- [Paper: Reconciling modern ML practice and the bias-variance trade-off](https://arxiv.org/abs/1912.02292)

## Image Classification with DNNs


Most machine learning workflows involve working with data, creating models, optimizing model parameters, and saving the trained models. This section introduces you to a complete ML workflow implemented in PyTorch, with links to learn more about each of these concepts.

We will use the FashionMNIST dataset to train a neural network that predicts if an input image belongs to one of the following classes: T-shirt/top, Trouser, Pullover, Dress, Coat, Sandal, Shirt, Sneaker, Bag, or Ankle boot.

### Working with Data

PyTorch has two primitives to work with data: `torch.utils.data.DataLoader` and `torch.utils.data.Dataset`. `Dataset` stores the samples and their corresponding labels, and `DataLoader` wraps an iterable around the `Dataset`.

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import Compose, ToTensor, Lambda
import matplotlib.pyplot as plt

PyTorch offers domain-specific libraries such as TorchText, TorchVision, and TorchAudio, all of which include datasets. For this tutorial, we will be using a TorchVision dataset.

The `torchvision.datasets` module contains `Dataset` objects for many real-world vision data like CIFAR, COCO. In this tutorial, we use the `FashionMNIST` dataset. Every TorchVision `Dataset` includes two arguments: `transform` and `target_transform`, to modify the samples and labels respectively.

In [ ]:
# Download training data from open datasets.
training_data = datasets.FashionMNIST(
    root="data_scratch",
    train=True,
    download=True,
    transform=Compose([
      ToTensor(),
      Lambda(lambda x: torch.flatten(x, start_dim=0))
    ]),
)

# Download test data from open datasets.
test_data = datasets.FashionMNIST(
    root="data_scratch",
    train=False,
    download=True,
    transform=Compose([
      ToTensor(),
      Lambda(lambda x: torch.flatten(x, start_dim=0))
    ]),
)

We pass the `Dataset` as an argument to `DataLoader`. This wraps an iterable over our dataset, and supports automatic batching, sampling, shuffling and multiprocess data loading. Here we define a batch size of 64, i.e. each element in the dataloader iterable will return a batch of 64 features and labels.

In [ ]:
batch_size = 64

# Create data loaders.
train_dataloader = DataLoader(training_data, batch_size=batch_size, pin_memory=True)
test_dataloader = DataLoader(test_data, batch_size=batch_size, pin_memory=True)

for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

We can also have a quick peek at the data

In [ ]:
images, labels = next(iter(train_dataloader))
print('Shape of input tensor:', list(images.shape))
ii = torch.reshape(images[0],(28,28))
plt.imshow(ii, cmap='gray')
plt.show()
print('Label: ', int(labels[0]))

### Creating Models
As we have in previous sections when learning Boolean functions, to define a neural network in PyTorch we create a class that inherits from `nn.Module`. We define the layers of the network in the `__init__` function (the constructor) and specify how data will pass through the network in the forward function. To accelerate operations in the neural network, we move it to the GPU if available.

In [ ]:
# Get cpu or gpu device for training.
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")

net = DeepNeuralNet(input_width=28 * 28, hidden_layer_profile=[512, 512], output_width=10).to(device)
print(net)

### Optimising Model Parameters

As illustrated before, to train a model, we need a loss function and an optimiser.

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimiser = torch.optim.SGD(net.parameters(), lr=1e-3)

The training loop from above will serve us well even in our current tasks. We also check the model’s performance against the test dataset to ensure it is learning -- in order to do so, we re-define the `testing_loop` function we used to learn Boolean functions in the new context of image classification.

In [ ]:
def testing_loop(dataloader, net,):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    net.eval()
    correct = 0

    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = net(X)
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    return correct / size

The training process is conducted over several iterations (epochs). During each epoch, the model learns parameters to make better predictions. We print the model's accuracy and loss at each epoch; we would like to see the accuracy increase and the loss decrease with every epoch.

In [ ]:
epochs = 5
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    training_loop(train_dataloader, net, loss_fn, optimiser)
    validation_accuracy = testing_loop(train_dataloader, net)
    print(f"Validation Accuracy: {validation_accuracy:.2%}\n")
print("Training Done!")

testing_accuracy = testing_loop(test_dataloader, net)
print(f"\nTest Accuracy: {testing_accuracy:.2%}")

We get an accuracy around 65 % which is much better than the 10 % we would get from random guessing.

**Exercise.** Play around with different learning setups. Modifying just the learning rate and the number of epochs, how high can you take the validation accuracy? While doing so, do you also observe similar improvements in the test accuracy? (Remember that you can adjust the verbosity level not to have to read through long listings).

In [ ]:
net = DeepNeuralNet(input_width=28 * 28, hidden_layer_profile=[512, 512], output_width=10).to(device)
optimiser = torch.optim.SGD(net.parameters(), lr=1e-3)
train(train_dataloader, net, loss_fn, optimiser, epochs=5, epoch_frequency=1, verbosity=2)

**Exercise.** Play around with the architecture of the neural network that you train. Adding more layers and more neurons, can you take the test performance even higher than in the previous exercise?

In [ ]:
# define the architecture of your neural network
best_net = DeepNeuralNet(input_width=28 * 28, hidden_layer_profile=[ None ], output_width=10).to(device)
optimiser = torch.optim.SGD(best_net.parameters(), lr=1e-3)
print(best_net)

# train it
train(train_dataloader, best_net, loss_fn, optimiser, epochs, verbosity=3)

# test it
testing_accuracy = testing_loop(test_dataloader, best_net)
print(f"\nTest Accuracy: {testing_accuracy:.2%}")

### Saving and Loading Models
Quite often you want to save your model, either to be later deployed in practice (on a website or in a mobile device, for example), or to be able to evaluate it later, in a different workflow. A common way to save a model is to serialise the internal state dictionary (containing the model parameters).

In [ ]:
torch.save(net.state_dict(), f"data_scratch/model.pth")
print(f"Saved PyTorch Model State to data_scratch/model.pth")

The process for loading a model includes re-creating the model structure and loading the state dictionary into it.

In [ ]:
model = DeepNeuralNet(28 * 28, [512, 512], 10)
model.load_state_dict(torch.load(f"data_scratch/model.pth"))

This model can now be used to make predictions.

In [ ]:
classes = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

model.eval()
x, y = test_data[0][0], test_data[0][1]
with torch.no_grad():
    pred = model(x)
    predicted, actual = classes[pred.argmax(0)], classes[y]
    print(f'Predicted: "{predicted}", Actual: "{actual}"')